In [2]:
#Tải các thư viện Python dùng cho việc xử lý dữ liệu
import pandas as pd
import numpy as np
print("Đã import thư viện thành công!")

Đã import thư viện thành công!


In [3]:
# Đọc file dữ liệu gốc từ thư mục data
data_path = "../data/raw/RT_IoT2022"  

df = pd.read_csv(data_path)

# Kiểm tra 5 dòng đầu tiên để xem dữ liệu mặt mũi ra sao
print(f"Kích thước ban đầu của dữ liệu: Dòng = {df.shape[0]}, Cột = {df.shape[1]}")
df.head()

Kích thước ban đầu của dữ liệu: Dòng = 123117, Cột = 85


,Unnamed: 0,id.orig_p,id.resp_p,proto,service,flow_duration,fwd_pkts_tot,bwd_pkts_tot,fwd_data_pkts_tot,bwd_data_pkts_tot,...,active.std,idle.min,idle.max,idle.tot,idle.avg,idle.std,fwd_init_window_size,bwd_init_window_size,fwd_last_window_size,Attack_type
0,0,38667,1883,tcp,mqtt,32.011598,9,5,3,3,...,0.0,2.972918e+07,2.972918e+07,2.972918e+07,2.972918e+07,0.0,64240,26847,502,MQTT_Publish
1,1,51143,1883,tcp,mqtt,31.883584,9,5,3,3,...,0.0,2.985528e+07,2.985528e+07,2.985528e+07,2.985528e+07,0.0,64240,26847,502,MQTT_Publish
2,2,44761,1883,tcp,mqtt,32.124053,9,5,3,3,...,0.0,2.984215e+07,2.984215e+07,2.984215e+07,2.984215e+07,0.0,64240,26847,502,MQTT_Publish
3,3,60893,1883,tcp,mqtt,31.961063,9,5,3,3,...,0.0,2.991377e+07,2.991377e+07,2.991377e+07,2.991377e+07,0.0,64240,26847,502,MQTT_Publish
4,4,51087,1883,tcp,mqtt,31.902362,9,5,3,3,...,0.0,2.981470e+07,2.981470e+07,2.981470e+07,2.981470e+07,0.0,64240,26847,502,MQTT_Publish


In [4]:
# Xóa cột 'Unnamed: 0' nếu nó xuất hiện do lúc lưu file bị thừa index cũ
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])
    print("Đã xóa cột 'Unnamed: 0' thừa.")
else:
    print("Không tìm thấy cột 'Unnamed: 0', dữ liệu sạch sẽ.")

Đã xóa cột 'Unnamed: 0' thừa.


In [5]:
# Kiểm tra và xóa các dòng dữ liệu bị trùng lặp hoàn toàn
print(f"Số dòng trước khi xóa trùng: {df.shape[0]}")

df = df.drop_duplicates()

print(f"Số dòng sau khi xóa trùng: {df.shape[0]}")

Số dòng trước khi xóa trùng: 123117
Số dòng sau khi xóa trùng: 117922


In [6]:
# Đếm lại số mẫu từng nhãn sau khi xóa trùng để biết lớp nào bị giảm
print(df['Attack_type'].value_counts()) 

Attack_type
DOS_SYN_Hping                 90089
Thing_Speak                    7654
ARP_poisioning                 7625
MQTT_Publish                   4142
NMAP_UDP_SCAN                  2584
NMAP_XMAS_TREE_SCAN            2010
NMAP_OS_DETECTION              2000
NMAP_TCP_scan                  1002
DDOS_Slowloris                  533
Wipro_bulb                      219
Metasploit_Brute_Force_SSH       36
NMAP_FIN_SCAN                    28
Name: count, dtype: int64


In [7]:
# Mã hóa nhãn 'Attack_type' sang dạng số nguyên
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
target_column = 'Attack_type'

if target_column in df.columns:
    if not pd.api.types.is_numeric_dtype(df[target_column]):
        df[target_column] = encoder.fit_transform(df[target_column])
        print(f"Đã mã hóa thành công cột mục tiêu '{target_column}' sang dạng số.")
    else:
        print("Cột mục tiêu đã ở dạng số từ trước.")
    
    class_mapping = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))
    print("Danh sách các nhãn sau khi mã hóa:")
    print(class_mapping)
else:
    print(f"Không tìm thấy cột mục tiêu {target_column}!")

Đã mã hóa thành công cột mục tiêu 'Attack_type' sang dạng số.
Danh sách các nhãn sau khi mã hóa:
{'ARP_poisioning': np.int64(0), 'DDOS_Slowloris': np.int64(1), 'DOS_SYN_Hping': np.int64(2), 'MQTT_Publish': np.int64(3), 'Metasploit_Brute_Force_SSH': np.int64(4), 'NMAP_FIN_SCAN': np.int64(5), 'NMAP_OS_DETECTION': np.int64(6), 'NMAP_TCP_scan': np.int64(7), 'NMAP_UDP_SCAN': np.int64(8), 'NMAP_XMAS_TREE_SCAN': np.int64(9), 'Thing_Speak': np.int64(10), 'Wipro_bulb': np.int64(11)}


In [8]:
from sklearn.model_selection import train_test_split

# Tách đặc trưng X và nhãn y
X = df.drop(columns=['Attack_type'])
y = df['Attack_type']

# One-Hot cho proto và service trước khi chia tập
X = pd.get_dummies(X, columns=['proto', 'service'], dtype=int)

# Chia 80% train, 20% test, giữ nguyên tỷ lệ từng nhãn (stratify)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Kích thước ban đầu sau khi tách & One-Hot: Train: {X_train.shape} Test: {X_test.shape}")

Kích thước ban đầu sau khi tách & One-Hot: Train: (94337, 94) Test: (23585, 94)


In [9]:
# Chỉ lấy các cột số gốc, bỏ qua các cột One-Hot (proto_, service_)
cot_goc = [c for c in X_train.columns if not c.startswith(('proto_', 'service_'))]

# Tính hệ số tương quan tuyệt đối chỉ trên tập train
tuong_quan = X_train[cot_goc].corr().abs()

# Giữ nửa tam giác trên để mỗi cặp chỉ xuất hiện một lần
nua_tren = tuong_quan.where(np.triu(np.ones(tuong_quan.shape), k=1).astype(bool))

# Cột nào có hệ số lớn hơn 0.95 với một cột đứng trước thì bị loại
cot_bo = [c for c in nua_tren.columns if any(nua_tren[c] > 0.95)]

print(f"Số cột bị loại bỏ do tương quan > 0.95 (tính trên tập Train): {len(cot_bo)}")

# Bỏ cùng các cột đó khỏi cả X_train và X_test
X_train = X_train.drop(columns=cot_bo)
X_test = X_test.drop(columns=cot_bo)

print(f"Số cột còn lại sau khi lọc đa cộng tuyến: Train: {X_train.shape}, Test: {X_test.shape}")

Số cột bị loại bỏ do tương quan > 0.95 (tính trên tập Train): 24
Số cột còn lại sau khi lọc đa cộng tuyến: Train: (94337, 70), Test: (23585, 70)


In [10]:
# Kiểm tra giá trị thiếu (missing values) trên tập train và test
missing_train = X_train.isnull().sum().sum()
missing_test = X_test.isnull().sum().sum()
print(f"Tổng số giá trị thiếu trong X_train: {missing_train} | X_test: {missing_test}")

Tổng số giá trị thiếu trong X_train: 0 | X_test: 0


In [11]:
from sklearn.preprocessing import StandardScaler

# Tạo bộ chuẩn hóa
scaler = StandardScaler()

# Học trung bình và độ lệch chuẩn CHỈ TRÊN TẬP TRAIN, rồi chuẩn hóa train
X_train_scaled = scaler.fit_transform(X_train)

# Dùng lại đúng các thông số đã học ở train để chuẩn hóa test (tránh rò rỉ dữ liệu)
X_test_scaled = scaler.transform(X_test)

In [12]:
import os, joblib

# Tạo thư mục lưu nếu chưa có
os.makedirs("../data/processed", exist_ok=True)

# Lưu bản chưa chuẩn hóa (dùng cho các mô hình cây như Decision Tree, Random Forest)
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

# Lưu bộ chuẩn hóa (dùng cho KNN, SVM, MLP, Logistic Regression)
joblib.dump(scaler, "../data/processed/scaler.pkl")

# Lưu ánh xạ nhãn đầy đủ để vẽ Confusion Matrix
pd.Series(encoder.classes_).to_csv("../data/processed/label_names.csv", index=False)

In [13]:
pd.Series(cot_bo).to_csv("../data/processed/dropped_corr_cols.csv", index=False)